<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB10_A_Complete_ML_Project_Yacht_Hull_Resistance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB10 · Class 10 — A Complete ML Project: Predicting Yacht Hull Resistance**

## Block 2: AI — Machine Learning (closing)

`NB07`–`NB09` each focused on one piece of the Machine Learning toolbox: the core workflow, specific algorithms, unsupervised learning. This class closes Block 2 by putting **every piece together into one complete project**, start to finish, on a dataset we haven't touched yet: the classic **[Yacht Hydrodynamics](https://archive.ics.uci.edu/dataset/243/yacht+hydrodynamics)** dataset (308 real towing-tank experiments on sailing yacht hulls, from the Technical University of Delft), already mirrored in this repository at [`Datasets/yacht_hydrodynamics.data`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/yacht_hydrodynamics.data).

**The problem**: given a hull's geometry (5 dimensionless coefficients) and its speed (via the [Froude number](https://en.wikipedia.org/wiki/Froude_number)), predict its **residuary resistance** — the drag a hull experiences beyond friction, and a central quantity in ship design.

### Learning objectives

By the end of this class, students will be able to:
- Describe the stages of a real ML project, from problem framing to a deployable model.
- Perform basic data curation and quality checks before modeling.
- Use correlation and feature importance together to reason about feature selection.
- Build and tune a full `Pipeline` with `GridSearchCV`, evaluating a final model only once, at the end.
- Read a learning curve to diagnose whether a model would benefit from more data or is close to its ceiling.
- Save a trained model to disk and reload it — the last step before a model could be used operationally.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap of NB01–NB09, today's roadmap | 5 min | Theory |
| 2 | The ML project workflow: framing a project end to end | 10 min | Theory |
| 3 | Loading and exploring the real dataset (yacht hull resistance) | 15 min | Practice |
| 4 | Data curation and quality checks | 10 min | Theory + Practice |
| 5 | Feature selection: correlation and importance together | 15 min | Practice |
| 6 | Building a model comparison pipeline | 20 min | Practice |
| 7 | Hyperparameter tuning with `GridSearchCV` | 15 min | Practice |
| 8 | Final evaluation: residuals and a learning curve | 20 min | Practice |
| 9 | Saving and reloading a trained model | 5 min | Practice |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB01`–`NB02`**: AI history, Python/Colab/NumPy/Pandas essentials.
- **`NB07`**: the supervised ML workflow, metrics, a classifier and a regressor, data leakage.
- **`NB08`**: classification algorithms in depth — trees, ensembles, SVM, leakage-safe pipelines, tuning.
- **`NB09`**: unsupervised learning — K-Means clustering, PCA.
- **`NB10`** (today): everything above, combined into one complete project on a brand-new real dataset.

This closes **Block 2 — AI: Machine Learning**. `NB11` opens Block 3 (Deep Learning).

---

## 2. The ML project workflow

Every real ML project — not just today's toy example — follows roughly the same stages:

| Stage | Question it answers | Where we've seen it |
|---|---|---|
| **1. Problem framing** | What exactly are we predicting, and why does it matter? | Today: predict residuary resistance from hull geometry |
| **2. Data collection** | Where does the data come from, can we trust it? | `!wget` from a mirrored, real, published dataset |
| **3. Exploration (EDA)** | What does the data actually look like? | `NB02`–`NB09`: `head`, `describe`, `groupby`, `.corr()`, plots |
| **4. Data curation** | Is it clean enough to model? | New today: duplicates, ranges, missing values, sanity checks |
| **5. Feature selection** | Which inputs actually help? | New today: correlation + model-based importance together |
| **6. Modeling** | Which algorithm(s) fit the problem? | `NB07`/`NB08`: Linear Regression, Random Forest, and friends |
| **7. Tuning** | Can we do better than default settings? | `NB08` introduced `GridSearchCV`; today we use it for real |
| **8. Final evaluation** | How good is the *final* model, honestly? | Test set touched exactly once, at the very end |
| **9. Persistence / deployment** | How would this model actually get used? | New today: saving and reloading a trained model |

The two genuinely new stages today are **4** (data curation) and **9** (persistence) — everything else builds directly on tools from `NB07`–`NB09`.

> **Further reading**: [CRISP-DM, a standard data-mining process model (Wikipedia)](https://en.wikipedia.org/wiki/Cross-industry_standard_process_for_data_mining).

---

## 3. Loading and exploring the real dataset

The **[Yacht Hydrodynamics](https://archive.ics.uci.edu/dataset/243/yacht+hydrodynamics)** dataset comes from towing-tank tests of 22 sailing yacht hull forms at Delft, at varying speeds — 308 real experimental measurements in total. Six numeric inputs describe hull geometry and speed; the seventh column is the target.

In [ ]:
!wget -q -O yacht.data https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/yacht_hydrodynamics.data

import pandas as pd

columns = [
    "LongPos_COB",       # longitudinal position of the center of buoyancy
    "Prismatic_Coeff",   # prismatic coefficient
    "LengthDisp_Ratio",  # length-displacement ratio
    "BeamDraft_Ratio",   # beam-draught ratio
    "LengthBeam_Ratio",  # length-beam ratio
    "Froude_Number",     # Froude number (speed)
    "Residuary_Resistance",  # target: residuary resistance per unit weight of displacement
]
yacht = pd.read_csv("yacht.data", sep=r"\s+", names=columns)
print(yacht.shape)
yacht.head()

The first five columns are fixed per hull design (they don't change with speed); `Froude_Number` varies because each hull was tested at multiple speeds — that's why 22 hulls produce 308 rows.

In [ ]:
yacht.describe()

---

## 4. Data curation and quality checks

Before modeling anything, a few quick, standard checks — the same questions any real dataset deserves, regardless of how clean it looks:

In [ ]:
print("missing values per column:")
print(yacht.isna().sum())
print()
print("duplicate rows:", yacht.duplicated().sum())
print()
print("Froude number range:", yacht["Froude_Number"].min(), "-", yacht["Froude_Number"].max())
print("Residuary resistance range:", yacht["Residuary_Resistance"].min(), "-", yacht["Residuary_Resistance"].max())

**Read your own output**: a real Froude number for a displacement hull should be a small positive value (roughly 0.1–0.5 here); residuary resistance should be non-negative. If either range looked physically impossible (negative resistance, Froude numbers in the thousands), `that would be a sign of a data entry error or a units mismatch` — worth catching *before* a model quietly learns to fit nonsense.

> **Further reading**: [Data curation (Wikipedia)](https://en.wikipedia.org/wiki/Data_curation).

---

## 5. Feature selection: correlation and importance together

With only 6 candidate features, we don't strictly need to drop any for this dataset — but the *reasoning* for feature selection matters even here, and will matter more on larger real datasets. Two complementary views:

1. **Correlation with the target** — a fast, model-free first look.
2. **Model-based feature importance** (as in `NB08`) — captures non-linear relationships correlation alone would miss.

In [ ]:
yacht.corr()["Residuary_Resistance"].sort_values(ascending=False)

`Froude_Number` should stand out as by far the strongest linear correlate — makes physical sense, since resistance grows sharply (non-linearly) with speed. The five hull-geometry columns correlate far more weakly *individually* — but that doesn't mean they're useless: resistance depends on how hull shape and speed **interact**, which a simple pairwise correlation can't capture. Let's check with a model that *can* capture interactions:

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = yacht.drop(columns="Residuary_Resistance")
y = yacht["Residuary_Resistance"]

rf_importance_check = RandomForestRegressor(n_estimators=300, random_state=42)
rf_importance_check.fit(X, y)

pd.Series(rf_importance_check.feature_importances_, index=X.columns).sort_values(ascending=False)

**Compare the two rankings**: does `Froude_Number` still dominate? Do any hull-geometry columns rank noticeably higher here than in the plain correlation above — evidence that they matter mainly *in combination* with speed, not on their own? We'll keep all 6 features going forward, but `this is exactly the reasoning you'd use to justify dropping a column on a larger, messier dataset`.

> **Further reading**: [Feature selection (Wikipedia)](https://en.wikipedia.org/wiki/Feature_selection).

---

## 6. Building a model comparison pipeline

Split the data once, hold the test set aside for the *entire rest of this notebook*, and compare three regressors — a linear baseline plus two ensembles, echoing the bagging-vs-boosting comparison from `NB08` — each in a scaling `Pipeline` for a fair, leakage-safe comparison via cross-validation on the training data only.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, " Test (untouched until Part 8):", X_test.shape)

Compare Linear Regression, Random Forest, and Gradient Boosting — a *sequential*, boosting-style ensemble for regression, the regression counterpart of `NB08`'s AdaBoost:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in candidate_models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2")
    cv_results[name] = scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)

The table above shows mean R², hiding how *consistent* each model is across folds. A boxplot shows both at once:

In [ ]:
pd.DataFrame(cv_results).boxplot(figsize=(8, 5))
plt.ylabel("Cross-validated R2")
plt.title("Model comparison — Yacht Hull Resistance")
plt.show()


**Interpret your own results**: does either ensemble clearly beat plain Linear Regression? If `Froude_Number`'s relationship with resistance is strongly non-linear (as the physics suggests), `the ensembles should have a real advantage here` — unlike in a dataset that's closer to linear, where the simpler model can be just as good. We'll take the best-performing model into Part 7 for tuning.

> **Further reading**: [`sklearn.ensemble.GradientBoostingRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html).

**Try it yourself**: Part 5 argued that a hull-geometry column contributing little on its own could still matter, or might genuinely be droppable. Test it directly — drop the weakest column by importance (`LengthDisp_Ratio`) and re-run Random Forest's cross-validation. Does removing it help, hurt, or make no real difference?

In [ ]:
weakest_feature = "LengthDisp_Ratio"

pipe_full = Pipeline([("scaler", StandardScaler()), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])
pipe_reduced = Pipeline([("scaler", StandardScaler()), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])

cv_full = cross_val_score(pipe_full, X_train, y_train, cv=cv, scoring="r2")
cv_reduced = cross_val_score(pipe_reduced, X_train.drop(columns=[weakest_feature]), y_train, cv=cv, scoring="r2")

print(f"CV R2 with all 6 features: {cv_full.mean():.4f}")
print(f"CV R2 without '{weakest_feature}': {cv_reduced.mean():.4f}")


---

## 7. Hyperparameter tuning with `GridSearchCV`

Take the strongest model from Part 6 (assumed here to be **Random Forest** — change the code below if your results favored Gradient Boosting instead) and search over its main hyperparameters: `n_estimators` (how many trees), `max_depth` (how deep each tree can grow — recall the overfitting-vs-depth demo from `NB08`), and `min_samples_leaf` (a second, gentler brake on tree complexity).

In [ ]:
from sklearn.model_selection import GridSearchCV

tuning_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(random_state=42)),
])

param_grid = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(tuning_pipe, param_grid, cv=cv, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV R2:", round(grid.best_score_, 3))

Compare `grid.best_score_` to the plain Random Forest row from Part 6 — the grid search includes the untuned defaults as one of its candidates, so `tuning should match or beat them, never do worse`.

**Try it yourself**: `grid.best_params_` only shows the single winning configuration. Look at the top 5 from `grid.cv_results_` instead — are there other configurations nearly as good, perhaps simpler (fewer trees, shallower) or more stable (lower `std_test_score`)?

In [ ]:
results_grid = pd.DataFrame(grid.cv_results_)
top5 = results_grid.sort_values("mean_test_score", ascending=False).head(5)
top5[["param_model__n_estimators", "param_model__max_depth", "param_model__min_samples_leaf", "mean_test_score", "std_test_score"]]


---

## 8. Final evaluation: residuals and a learning curve

Now — and only now — do we touch the test set we set aside in Part 6.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test, y_pred), 3))
print("Test RMSE:", round(mean_squared_error(y_test, y_pred) ** 0.5, 3))
print("Test R2:", round(r2_score(y_test, y_pred), 3))

Before looking at residuals, the classic predicted-vs-actual view: points close to the diagonal are hulls the model predicted well.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
lims = [y.min(), y.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual residuary resistance")
plt.ylabel("Predicted residuary resistance")
plt.title("Final tuned model — predicted vs. actual")
plt.legend()
plt.show()


Compare this test R² to the cross-validated R² from Parts 6–7 — they should be close. A test score much worse than the cross-validated one would suggest the tuning process itself overfit to the training data (a subtler leakage risk, since `GridSearchCV` tries many configurations and could get lucky on one by chance).

A residual plot shows *where* the model struggles, not just an average error:

In [ ]:
import matplotlib.pyplot as plt

residuals = y_test - y_pred

plt.figure(figsize=(6, 5))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted residuary resistance")
plt.ylabel("Residual (actual − predicted)")
plt.title("Residual plot — final tuned model")
plt.show()

Residuals scattered evenly around zero, with no obvious pattern or funnel shape, suggest the model's errors are roughly consistent across the prediction range. A funnel shape (errors growing with predicted resistance) would suggest `the model is less reliable for high-resistance hulls` — genuinely useful to know before trusting it operationally.

One more diagnostic: a **learning curve** shows how performance changes as we give the model more training data — it tells us whether collecting more data would even help.

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=cv, scoring="r2",
    train_sizes=np.linspace(0.2, 1.0, 6), random_state=42,
)

plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training R2")
plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation R2")
plt.xlabel("Training set size")
plt.ylabel("R2 score")
plt.title("Learning curve")
plt.legend()
plt.show()

**Read your own curve**: if the validation curve is still rising and hasn't met the training curve by the right edge of the plot, more data would likely help. If both curves have flattened and converged, this model has roughly reached its ceiling on this feature set — more *data* wouldn't help much, but better *features* or a different *algorithm* might.

> **Further reading**: [`sklearn.model_selection.learning_curve` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html).

---

## 9. Saving and reloading a trained model

`Training a model is pointless if it only ever exists inside one notebook session`. `joblib` serializes a fitted scikit-learn object (including an entire `Pipeline`) to a file, so it can be reloaded — in a different notebook, a script, or a small web service — without retraining.

In [ ]:
import joblib

joblib.dump(best_model, "yacht_resistance_model.pkl")
print("Model saved.")

reloaded_model = joblib.load("yacht_resistance_model.pkl")
print("Reloaded model predicts:", reloaded_model.predict(X_test.iloc[[0]]))
print("Actual value was:       ", y_test.iloc[0])

That's the entire journey a real ML project takes: raw data in, a reusable, tested, saved model out.

> **Further reading**: [scikit-learn's guide to model persistence](https://scikit-learn.org/stable/model_persistence.html) · [`joblib` documentation](https://joblib.readthedocs.io/en/stable/).

---

## Class summary

- A real ML project follows a fairly consistent set of stages: frame the problem, get and curate the data, explore it, select features, model, tune, evaluate once at the end, and persist the result.
- Data curation (missing values, duplicates, physically-sane ranges) is a fast but essential check before modeling — especially on data you didn't collect yourself.
- Correlation and model-based feature importance answer different questions; using both gives a fuller picture than either alone, especially when features interact (hull shape × speed, here).
- `GridSearchCV` inside a `Pipeline` tunes hyperparameters without leaking test data into the process.
- The test set is touched exactly once, at the very end — and its score should roughly match cross-validation, or something went wrong earlier.
- A learning curve tells you whether more data would help; a residual plot tells you where a model's errors concentrate.
- `joblib` turns a trained model from a notebook-only artifact into something reusable elsewhere.

## For the next class (NB11)

We open **Block 3 — Deep Learning**: neural networks, from a single perceptron up to the building blocks of CNNs, filling the biggest content gap identified against the course's guía docente.

## Homework / Practice Ideas

1. Repeat Part 6's model comparison, but add `SVR` (Support Vector Regression, the regression counterpart of `NB08`'s `SVC`) to the candidates — how does it compare to the ensembles?
2. Extend the Part 7 grid search: add `model__max_features` as a fourth tuned hyperparameter — does it change the best configuration found?
3. In Part 8, identify the 5 test-set hulls with the largest residuals (by absolute value) — do they share anything in common (a particular Froude number range, or hull geometry)?
4. Re-run the Part 8 learning curve using only `Froude_Number` as a feature (dropping the 5 hull-geometry columns) — how much worse does the ceiling get, and what does that tell you about how much those 5 columns are actually contributing?
5. Load your saved `yacht_resistance_model.pkl` in a *fresh* Colab runtime (Runtime → Restart session) and confirm it still predicts correctly without re-running Parts 1–8 — this is the real test of whether persistence actually worked.

> ***As always: a tuned model is only as trustworthy as the data curation and honest final evaluation behind it — a high R² on a leaked test set is worse than a modest R² on a clean one.***
